![FL_Logo_FullColor_Horiz.png](FL_Logo_FullColor_Horiz.png)

# FiberLocator vs. FCC BDC Data
Updated: 2026-03-20  
Mike Iapaucci- Karen Pesca  
Vice President, CCMI  
miapalucci@ccmi.com  
env: gis

In [ ]:
pip install --upgrade pip

In [ ]:
!pip install folium

In [ ]:
!pip install geojson

In [ ]:
!pip install h3


In [ ]:
# First, check which environment you're using
import sys
print(f"Python executable: {sys.executable}")

# Try installing with pip ensuring it uses the correct Python
!{sys.executable} -m pip install folium

In [ ]:
import sys
!{sys.executable} -m pip install session_info

In [ ]:
!python -m pip install h3

In [ ]:
!which python
!pip list | grep h3

In [ ]:
# Try installing with pip ensuring it uses the correct Python
!{sys.executable} -m pip install matplotlib

In [ ]:
# Try installing with pip ensuring it uses the correct Python
!{sys.executable} -m pip install shapely

In [ ]:
# Try installing with pip ensuring it uses the correct Python
!{sys.executable} -m pip install geopandas

In [ ]:
# Try installing with pip ensuring it uses the correct Python
!{sys.executable} -m pip install geojson

In [ ]:
import csv
import folium
import h3
import json
import math
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pprint
pp = pprint.PrettyPrinter(indent=4)
import requests
import shapely.geometry as geometry

import geopandas as gpd
from geojson import Polygon, Feature, FeatureCollection, dump
from folium import features
from PIL import Image

In [ ]:
import os
os.environ['CONDA_DEFAULT_ENV']

In [ ]:
import sys
!{sys.executable} -m pip install session_info

In [ ]:
import session_info
session_info.show()

In [ ]:
!{sys.executable} -m pip install geopandas

In [ ]:
!{sys.executable} -m pip install folium

In [ ]:
!{sys.executable} -m pip install h3

In [ ]:
!{sys.executable} -m pip install geojson

## Step 1: Download data from FCC BDC  
https://broadbandmap.fcc.gov/data-download

In [ ]:
import os
import pandas as pd

# Option 1: Check your current working directory
print("Current working directory:", os.getcwd())


In [ ]:
# Fiber
tx_fiber = pd.read_csv("bdc_48_FibertothePremises_fixed_broadband_J25_03mar2026.csv")
tx_fiber.head()

In [ ]:
#Number of observations fiber premises
len (tx_fiber)

In [ ]:
#cable texas
tx_cable = pd.read_csv("bdc_48_Cable_fixed_broadband_J25_03mar2026.csv")
tx_cable.head()

In [ ]:
#number of observations cable
len(tx_cable)

In [ ]:
# Combine fiber and cable into one dataframe TEXAS
tx_fiber_cable = pd.concat([tx_fiber, tx_cable])
tx_fiber_cable.value_counts("technology").sort_values()

## Step 2: Filter data for counties of interest and business clients Odessa
Cities: Odessa TX (Ector County-FIPS= 48135, Midland County- FIPS= 48329)

In [ ]:
tx_fiber_cable_4 = tx_fiber_cable[tx_fiber_cable["block_geoid"].apply(lambda x: str(x)[:5] in ["48135","48329"])]

In [ ]:
print("All of Texas: ", tx_fiber_cable.shape[0])
print("Odessa: ", tx_fiber_cable_4.shape[0])

In [ ]:
tx_fiber_cable_4.value_counts("business_residential_code")

In [ ]:
# Remove the solely residential buildings (code=R)
tx_fiber_cable_4 = tx_fiber_cable_4[tx_fiber_cable_4["business_residential_code"] != "R"]
tx_fiber_cable_4.value_counts("business_residential_code")

In [ ]:
# Change codes to proper names
tx_fiber_cable_4["business_code_name"] = tx_fiber_cable_4["business_residential_code"].apply(lambda x: "Biz/Res" if x =="X" \
else "Biz")
tx_fiber_cable_4["technology_name"] = tx_fiber_cable_4["technology"].apply(lambda x: "Fiber" if x == 50 else "Cable")

In [ ]:
pd.crosstab(tx_fiber_cable_4["technology_name"], tx_fiber_cable_4["business_code_name"], margins=True, margins_name="Total")

## Step 3: Create summary data for use as attribute data

In [ ]:
tx_fiber_cable_4.value_counts("brand_name").sort_values(ascending=False)

### Create a Bar charts for brand_name as my 1st visualization


In [ ]:
# Count brands
brand_counts = tx_fiber_cable_4["brand_name"].value_counts()

# Take top 10
top_brands = brand_counts.head(10)

In [ ]:
#Bart chat

brand_counts = tx_fiber_cable_4["brand_name"].value_counts()
top_brands = brand_counts.head(10)

# color palette
colors = plt.cm.tab10(np.linspace(0, 1, len(top_brands)))

plt.figure(figsize=(12, 6))

top_brands.sort_values().plot(kind='barh', color=colors)

plt.title("Top 10 Broadband Providers in Odessa and Wichita Falls Texas (FCC Data)")
plt.xlabel("Number of Locations")
plt.ylabel("Provider")

# labels
for index, value in enumerate(top_brands.sort_values()):
    plt.text(value, index, f" {value}", va='center')

plt.tight_layout()
plt.show()

In [ ]:
tx_fiber_cable_4.head()

In [ ]:
tx_fiber_cable_4["h3_res8_id"].value_counts().head(10)

In [ ]:
# Potential for multiple providers in any given h3 hexagon.  Here the first hex in the list has 3 providers with a total of
# 2354 locations.
hex_1tx = tx_fiber_cable_4[["brand_name", "h3_res8_id"]][tx_fiber_cable_4["h3_res8_id"] == "8848d6d95dfffff"]
print(hex_1tx.value_counts("brand_name"))
print("Total locations in hex 1: ", len(hex_1tx["h3_res8_id"]))

In [ ]:
hex_1tx.columns

In [ ]:
print(tx_fiber_cable_4.columns.tolist())

In [ ]:
tx_attributes = tx_fiber_cable_4[["brand_name", "location_id", "h3_res8_id", "technology_name",\
                                "business_code_name"]].groupby(by=['h3_res8_id']).agg(lambda x: list(x))
tx_attributes.head()

In [ ]:
# Create unique counts of brand_name, location_id, technology_name, and business_code_name for each h3 hexagon.
tx_attributes["unique_brands"] = tx_attributes["brand_name"].apply(lambda x: list(set(x)))
tx_attributes["unique_brands_count"] = tx_attributes["unique_brands"].apply(lambda x: len(x))
tx_attributes["locations"] = tx_attributes["location_id"].apply(lambda x: len(x))
tx_attributes["technologies"] = tx_attributes["technology_name"].apply(lambda x: list(set(x)))
tx_attributes["biz_vs_biz/res"] = tx_attributes["business_code_name"].apply(lambda x: list(set(x)))

In [ ]:
tx_attributes.head()

In [ ]:
hex_provider_counttx = tx_attributes['unique_brands_count'].value_counts().sort_values(ascending=False)
print(hex_provider_counttx)
print("Total hexagons: ", np.sum(hex_provider_counttx))

## Step 4: Create H3 hexagon geometry and key attributes in GeoJSON for mapping

In [ ]:
print(tx_attributes.columns.tolist())

In [ ]:
tx_attributes = tx_attributes.reset_index()

In [ ]:
tx_attributes.head()

In [ ]:
from shapely.geometry import Polygon

def add_geometry(row):
    boundary = h3.cell_to_boundary(row["h3_res8_id"])  # returns (lat, lng)
    points = [(lng, lat) for lat, lng in boundary]     # convert to (lng, lat)
    return Polygon(points)

tx_attributes["geometry"] = tx_attributes.apply(add_geometry, axis=1)

In [ ]:
tx_attributes['geometry'] = (tx_attributes.apply(add_geometry, axis=1))

In [ ]:
tx_attributes.iloc[0]

In [ ]:
tx_attributes = tx_attributes[["h3_res8_id", "location_id", "unique_brands", "unique_brands_count", "locations",
                              "technologies", "biz_vs_biz/res", "geometry"]]
tx_attributes["location_id"] = tx_attributes["location_id"].astype(str)
tx_attributes["unique_brands"] = tx_attributes["unique_brands"].astype(str)
tx_attributes["technologies"] = tx_attributes["technologies"].astype(str)
tx_attributes["biz_vs_biz/res"] = tx_attributes["biz_vs_biz/res"].astype(str)
tx_attributes.head()

In [ ]:
tx_geojson = gpd.GeoDataFrame(tx_attributes)

In [ ]:
def hexagons_dataframe_to_geojson(df_hex, hex_id_field, geometry_field, value_field, file_output = None):

    list_features = []

    for i, row in df_hex.iterrows():
        feature = Feature(geometry = row[geometry_field],
                          id = row[hex_id_field],
                          properties = {"value": row[value_field]})
        list_features.append(feature)

    feat_collection = FeatureCollection(list_features)

    if file_output is not None:
        with open(file_output, "w") as f:
            json.dump(feat_collection, f)

    else :
      return feat_collection

In [ ]:
tx_geojson.head()

In [ ]:
tx_geojson.to_file("tx_bdc.geojson", driver="GeoJSON")

## Step 5: Mapping the data

In [ ]:
import geojson
with open("tx_Counties.geojson") as f:
    counties = geojson.load(f)

In [ ]:
lat = 31.8457
lon = -102.3676
zoom_start = 10

m = folium.Map(location=[lat, lon], zoom_start=zoom_start)

geojson_data = counties
folium.GeoJson(geojson_data,
               name="County Boundaries",
               style_function=lambda feature:{"fillColor": "#85C1E9",
                                             "color": "black",
                                             "weight": 5,
                                             "dashArray": "5, 5",}
              ).add_to(m)

bdc = folium.GeoJson(
    'tx_bdc.geojson',
    name = 'FCC BDC Data',
    show=False,
).add_to(m)

folium.LayerControl(position="bottomright", collapsed=False).add_to(m)

m

In [ ]:
import geojson
with open("tx_bdc.geojson") as f:
    tx_bdc = geojson.load(f)

In [ ]:
# Odessa center
odessa_lat = 31.8457
odessa_lon = -102.3676



In [ ]:
tx_bdc['features'][0]

In [ ]:
zoom_start = 10

m = folium.Map(location=[lat, lon],
               zoom_start=zoom_start)

geojson_data = counties
folium.GeoJson(geojson_data,
               name="County Boundaries",
               style_function=lambda feature:{"fillColor": "#85C1E9",
                                             "color": "black",
                                             "weight": 2,
                                             "dashArray": "5, 5",}
              ).add_to(m)

choropleth = folium.Choropleth(
    geo_data = tx_bdc,
    name = 'FCC BDC Data',
    data = tx_geojson,
    columns = ['h3_res8_id', 'locations'],
    key_on = 'feature.properties.h3_res8_id',
    fill_color = 'YlGn',
    fill_opacity = 0.7,
    #line_color = 'Black',
    line_opaciy = 0.1,
    legend_name = 'Number of BSLs Served',
    highlight = True,
).add_to(m)

choropleth.geojson.add_child(
    folium.features.GeoJsonTooltip(['locations', 'biz_vs_biz/res', 'unique_brands'], labels=True)
)

folium.LayerControl(position="bottomright", collapsed=False).add_to(m)

m

In [ ]:
lat = 31.8457
lon = -102.3676
zoom_start = 8

m = folium.Map(
    location=[lat, lon],
    zoom_start=zoom_start,
    tiles="CartoDB positron"
)

folium.GeoJson(
    counties,
    name="County Boundaries",
    style_function=lambda feature: {
        "fillColor": "#85C1E9",
        "color": "black",
        "weight": 2,
        "dashArray": "5, 5",
        "fillOpacity": 0.1
    }
).add_to(m)

folium.GeoJson(
    "tx_bdc.geojson",
    name="FCC BDC Data",
    show=True,
    style_function=lambda feature: {
        "fillColor": "#2166ac",
        "color": "#2166ac",
        "weight": 1,
        "fillOpacity": 0.45
    }
).add_to(m)

folium.LayerControl(position="bottomright", collapsed=False).add_to(m)

m

# Mapping the hex for odessa
## FCC BDC H3 hexagons for the selected Texas counties
- where the data exists

- how coverage is spatially distributed

- the concentration pattern around Odessa and Wichita Falls

In [ ]:
#import h3
#import geopandas as gpd
#from shapely.geometry import Polygon
#import folium
#import geojson

# Step 1: Correct geometry creation
#def add_geometry(row):
    #points = h3.cell_to_boundary(row["h3_res8_id"])
    #points_lonlat = [(lng, lat) for lat, lng in points]
    #return Polygon(points_lonlat)

# Step 2: Rebuild GeoDataFrame
tx_geojson = gpd.GeoDataFrame(
    tx_attributes,
    geometry=tx_attributes.apply(add_geometry, axis=1)
)

tx_geojson = tx_geojson.set_crs("EPSG:4326", allow_override=True)

# Step 3: Export corrected GeoJSON
tx_geojson.to_file("tx_bdc.geojson", driver="GeoJSON")

# Step 4: Load GeoJSON and map
with open("tx_bdc.geojson") as f:
    tx_hexes = geojson.load(f)

lat = 31.8
lon = -102.3
zoom_start = 10

m = folium.Map(location=[lat, lon], zoom_start=zoom_start, tiles="CartoDB positron")

folium.GeoJson(
    tx_hexes,
    name="FCC BDC Texas",
    style_function=lambda feature: {
        "fillColor": "blue",
        "color": "blue",
        "weight": 0.5,
        "fillOpacity": 0.4
    }
).add_to(m)

folium.LayerControl(position="bottomright", collapsed=False).add_to(m)

m

## improvement map colors

This map shows the H3 hexagons generated from the FCC BDC data for the selected Texas study areas. The spatial distribution highlights clusters of broadband serviceable locations around Odessa and Wichita Falls, confirming that the filtered dataset is correctly mapped to the areas of interest.

The color of each hexagon represents the number of broadband serviceable locations within that area. Darker shades indicate a higher concentration of locations, while lighter shades represent areas with fewer locations. This allows for a visual understanding of how broadband coverage density varies across the selected regions.

In [ ]:
def get_color(value):
    if value >= 5000:
        return "#08306b"
    elif value >= 2000:
        return "#2171b5"
    elif value >= 1000:
        return "#6baed6"
    elif value >= 500:
        return "#9ecae1"
    else:
        return "#c6dbef"

m = folium.Map(location=[31.8, -102.3], zoom_start=10, tiles="CartoDB positron")

folium.GeoJson(
    tx_hexes,
    name="FCC BDC Texas",
    style_function=lambda feature: {
        "fillColor": get_color(feature["properties"]["locations"]),
        "color": "black",
        "weight": 0.3,
        "fillOpacity": 0.6
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["h3_res8_id", "locations", "unique_brands_count"],
        aliases=["Hex ID:", "Locations:", "Providers:"]
    )
).add_to(m)

folium.LayerControl(position="bottomright", collapsed=False).add_to(m)

m

## Step 6: Add FiberLocator data to the map

In [ ]:
url = 'https://app.fiberlocator.com/rest/'

In [ ]:
creds = {'login': '', 'password': ''}

In [ ]:
endpoint = f'{url}login'
r = requests.post(endpoint, data = creds)
cookies = r.cookies
print(endpoint)
print(r)

In [ ]:
endpoint = f'{url}/token'
get_token = requests.get(endpoint, cookies=r.cookies)
token = get_token.json()['result']['token']

In [ ]:
zoom_start = 10
lat = 31.8
lon = -102.3
m = folium.Map(location=[lat, lon],
               zoom_start=zoom_start,
               )

geojson_data = counties
folium.GeoJson(geojson_data,
               name="County Boundaries",
               style_function=lambda feature:{"fillColor": "#85C1E9",
                                             "color": "black",
                                             "weight": 2,
                                             "dashArray": "5, 5",},
               ).add_to(m)

choropleth = folium.Choropleth(
    geo_data = tx_bdc,
    name = 'FCC BDC Data',
    data = tx_geojson,
    columns = ['h3_res8_id', 'locations'],
    key_on = 'feature.properties.h3_res8_id',
    fill_color = 'YlGn',
    fill_opacity = 0.2,
    #line_color = 'Black',
    line_opacity = 0.1,
    legend_name = 'BDC Data',
    highlight = True,
    overlay=True,
    show=True,
).add_to(m)

choropleth.geojson.add_child(
    folium.features.GeoJsonTooltip(['locations', 'unique_brands'], labels=True)
)

fg=folium.FeatureGroup(name='Metro Networks', show=True)

folium.raster_layers.WmsTileLayer(url = f'{url}{token}'+ '/maps/wmts/metro/webmercator/{z}/{x}/{y}.png',
                                  layers='metro',
                                  min_zoom=zoom_start,
                                  fmt='image/png',
                                  attr='<a href=https://www.fiberlocator.com>| FiberLocator</a>',
                                  transparent=True,
                                  overlay=True,
                                  ).add_to(fg)
m.add_child(fg)

fg=folium.FeatureGroup(name='Long Haul Networks', show=True)

folium.raster_layers.WmsTileLayer(url = f'{url}{token}'+ '/maps/wmts/longhaul/webmercator/{z}/{x}/{y}.png',
                                  layers='longhaul',
                                  min_zoom=zoom_start,
                                  fmt='image/png',
                                  attr='<a href=https://www.fiberlocator.com>| FiberLocator</a>',
                                  transparent=True,
                                  overlay=True,
                                  ).add_to(fg)
m.add_child(fg)

fg=folium.FeatureGroup(name='Connected Buidlings', show=True)

folium.raster_layers.WmsTileLayer(url = f'{url}{token}'+ '/maps/wmts/lit_bldgs/webmercator/{z}/{x}/{y}.png',
                                  layers='longhaul',
                                  min_zoom=zoom_start,
                                  fmt='image/png',
                                  attr='<a href=https://www.fiberlocator.com>| FiberLocator</a>',
                                  transparent=True,
                                  overlay=True,
                                  ).add_to(fg)
m.add_child(fg)

folium.LayerControl(position="bottomright", collapsed=False).add_to(m)

m
 


In [ ]:
# Check if the first ID in your data exists in your GeoJSON
sample_id = tx_geojson['h3_res8_id'].iloc[0]
print(f"Looking for {sample_id} in GeoJSON...")

# Manually check the GeoJSON structure
import json
# If tx_bdc is a file path or dict, check the first feature:
print(tx_bdc['features'][0]['properties'])

In [ ]:
tx_geojson['h3_res8_id'] = tx_geojson['h3_res8_id'].astype(str)

In [ ]:
import json
# Assuming tx_bdc is your GeoJSON object
first_feature_props = tx_bdc['features'][0]['properties']
print(f"Available properties: {list(first_feature_props.keys())}")

In [ ]:
sample_id = tx_geojson['h3_res8_id'].iloc[0]
exists = any(f['properties'].get('h3_res8_id') == sample_id for f in tx_bdc['features'])
print(f"Is {sample_id} in the GeoJSON? {exists}")

In [ ]:
folium.GeoJson(tx_bdc).add_to(m)
m # Display map

In [ ]:
# Fixed FCC Choropleth
choropleth = folium.Choropleth(
    geo_data = tx_bdc,
    name = 'FCC BDC Data',
    data = tx_geojson,
    columns = ['h3_res8_id', 'locations'],
    key_on = 'feature.properties.h3_res8_id',
    fill_color = 'YlGn',
    fill_opacity = 0.5,      # Increased opacity to make it easier to see
    line_opacity = 0.2,      # Fixed typo: 'line_opaciy' -> 'line_opacity'
    legend_name = 'BDC Data',
    highlight = True,
    overlay=True,
    show=True,
).add_to(m)

# Fixed Connected Buildings Layer
fg_bldgs = folium.FeatureGroup(name='Connected Buildings', show=True)
folium.raster_layers.WmsTileLayer(
    url = f'{url}{token}/maps/wmts/lit_bldgs/webmercator/{{z}}/{{x}}/{{y}}.png',
    layers='lit_bldgs',      # Fixed: was 'longhaul'
    min_zoom=0,              # Changed from zoom_start so it doesn't disappear when zooming out
    fmt='image/png',
    attr='FiberLocator',
    transparent=True,
    overlay=True,
).add_to(fg_bldgs)
m.add_child(fg_bldgs)

In [ ]:
import folium
import json

# 1. Verify Data Linkage (The most common failure)
df_sample_id = str(tx_geojson['h3_res8_id'].iloc[0])
geojson_sample_id = str(tx_bdc['features'][0]['properties'].get('h3_res8_id'))

print(f"DataFrame ID Sample: '{df_sample_id}'")
print(f"GeoJSON ID Sample:   '{geojson_sample_id}'")

if df_sample_id != geojson_sample_id:
    print("⚠️ ALERT: IDs do not match! Check for leading zeros or data types.")

# 2. Re-initialize Map centered on Texas
m = folium.Map(location=[31.9686, -99.9018], zoom_start=6)

# 3. Add Choropleth with explicit reset
# Note: key_on must match the GeoJSON structure exactly
choropleth = folium.Choropleth(
    geo_data=tx_bdc,
    name='FCC BDC Data',
    data=tx_geojson,
    columns=['h3_res8_id', 'locations'],
    key_on='feature.properties.h3_res8_id', # Ensure this property exists!
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Broadband Locations',
    highlight=True
).add_to(m)

# 4. Corrected WMS Layer (FiberLocator)
# Note: Some WMS services require specific zoom levels to activate
folium.WmsTileLayer(
    url=f"{url}{token}/maps/wmts/lit_bldgs/webmercator/{{z}}/{{x}}/{{y}}.png",
    layers='lit_bldgs',
    fmt='image/png',
    attr='FiberLocator',
    transparent=True,
    name='FiberLocator Buildings',
    overlay=True,
    control=True,
    show=True
).add_to(m)

folium.LayerControl().add_to(m)

m

In [ ]:

import folium
m = folium.Map(location=[31.96, -99.90], zoom_start=6)
m

In [ ]:
folium.raster_layers.WmsTileLayer(
    url=f"{url}{token}/maps/wmts/lit_bldgs/webmercator/{{z}}/{{x}}/{{y}}.png",
    layers='lit_bldgs',
    fmt='image/png',
    transparent=True,
    attr='FiberLocator',
    name='FiberLocator Buildings',
    overlay=True,
    show=True
).add_to(m)
m

In [ ]:
m.save("SW_tx_Broadband.html")

## Step 7: Compare FiberLocator providers to BDC providers

In [ ]:
# Minimum bounding box incorporating the 4 counties
# Odessa area bbox
bbox = [-102.8, 31.5, -101.7, 32.3]


In [ ]:
zoom_start = 9
lat = 31.8
lon = -102.3
zoom_start = 9

m = folium.Map(location=[lat, lon],
               zoom_start=zoom_start)

geojson_data = counties
folium.GeoJson(geojson_data,
               name="County Boundaries",
               style_function=lambda feature:{"fillColor": "#85C1E9",
                                             "color": "black",
                                             "weight": 2,
                                             "dashArray": "5, 5",}
              ).add_to(m)

kw = {
    "color": "red",
    "fill": False,
    "weight": 5,
    "popup": "Minimum Bounding Box",
}
folium.Rectangle(
    # lat, lon
    bounds=[[bbox[1], bbox[0]], [bbox[3], bbox[2]]],
    line_join="miter",
    #dash_array="5,10",
    **kw,
).add_to(m)

#folium.LayerControl(position="bottomright", collapsed=False).add_to(m)

m



In [ ]:
zoom_start = 9
lat = 31.8
lon = -102.3
m = folium.Map(location=[lat, lon],
               zoom_start=zoom_start)

geojson_data = counties
folium.GeoJson(geojson_data,
               name="County Boundaries",
               style_function=lambda feature:{"fillColor": "#85C1E9",
                                             "color": "black",
                                             "weight": 2,
                                             "dashArray": "5, 5",}
              ).add_to(m)

kw = {
    "color": "red",
    "fill": False,
    "weight": 5,
    "popup": "Minimum Bounding Box",
}
folium.Rectangle(
    bounds=[[bbox[1], bbox[0]], [bbox[3], bbox[2]]],  # Fixed: Added 'bounds=' parameter and proper comma
    line_join="miter",
    dash_array="5,10",
    **kw,
).add_to(m)

folium.LayerControl(position="bottomright", collapsed=False).add_to(m)

m

In [ ]:
# Access all providers from FiberLocator within the bounding box.
# lon, lat

endpoint = f'{url}{token}/layers/inview/{bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]}/longhaul,metro,buildings,'
layers_inview = requests.get(endpoint)
pp.pprint(layers_inview.json())

In [ ]:
FLO_Layers = pd.DataFrame(layers_inview.json()["result"], columns=["layer_name"])
FLO_Layers_Unique = pd.DataFrame(FLO_Layers["layer_name"].unique(), columns=["layer_name"]).sort_values("layer_name")
print("Unique Layers: ", len(FLO_Layers_Unique))
FLO_Layers_Unique

In [ ]:
FLO_Layers_Unique.to_csv("FLO_Layers_Unique.csv", index=False)

In [ ]:
BDC_Providers_Unique = pd.DataFrame(tx_fiber_cable_4["brand_name"].unique(), columns=["brand_name"]).sort_values("brand_name")
print("Unique Providers: ", len(BDC_Providers_Unique))
BDC_Providers_Unique

In [ ]:
tx_fiber_cable_4["brand_name"].unique()

In [ ]:
#tx_fiber_cable_4[tx_fiber_cable_4["brand_name"] == "Smithville Communications"].head() #Indiana

In [ ]:
tx_fiber_cable_4[tx_fiber_cable_4["brand_name"] == "Spectrum"].head()

In [ ]:
tx_fiber_cable_4[tx_fiber_cable_4["brand_name"] == "AT&T"].head()

In [ ]:
tx_fiber_cable_4[tx_fiber_cable_4["brand_name"] == "Fusion"].head()